# LightGCN - Analyse et Exploration

Ce notebook explore les données, visualise le graphe, et analyse les résultats du modèle.

## 1. Chargement et Exploration des Données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Ajouter le chemin src
sys.path.insert(0, '../src')
from data_loader import MovieLensDataLoader
from graph_builder import BipartiteGraphBuilder

In [ ]:
# Charger les données
loader = MovieLensDataLoader('../data/raw/ml-latest-small')
ratings = loader.load_ratings()
movies = loader.load_movies()

print("\n=== Dataset MovieLens ===")
print(f"Nombre de ratings: {len(ratings)}")
print(f"Nombre d'utilisateurs: {ratings['userId'].nunique()}")
print(f"Nombre de films: {ratings['movieId'].nunique()}")
print(f"\nPremières lignes:")
print(ratings.head())

In [ ]:
# Statistiques de base
print("\n=== Statistiques des Ratings ===")
print(ratings['rating'].describe())

# Distribution des ratings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(ratings['rating'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des Ratings')

# Count plot
rating_counts = ratings['rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Nombre de Ratings')
axes[1].set_title('Compte des Ratings')

plt.tight_layout()
plt.show()

In [ ]:
# Activité des utilisateurs
user_activity = ratings.groupby('userId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution du nombre de ratings par utilisateur
axes[0].hist(user_activity, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Nombre de Ratings')
axes[0].set_ylabel('Nombre d\'Utilisateurs')
axes[0].set_title('Distribution de l\'Activité Utilisateur')
axes[0].set_yscale('log')

# Top 20 utilisateurs
top_users = user_activity.nlargest(20)
axes[1].barh(range(len(top_users)), top_users.values)
axes[1].set_yticks(range(len(top_users)))
axes[1].set_yticklabels(top_users.index)
axes[1].set_xlabel('Nombre de Ratings')
axes[1].set_title('Top 20 Utilisateurs')

plt.tight_layout()
plt.show()

print(f"\nActivité utilisateur:")
print(f"  Min ratings: {user_activity.min()}")
print(f"  Max ratings: {user_activity.max()}")
print(f"  Moyenne: {user_activity.mean():.1f}")
print(f"  Médiane: {user_activity.median():.1f}")

In [ ]:
# Popularité des films
movie_popularity = ratings.groupby('movieId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution du nombre de ratings par film
axes[0].hist(movie_popularity, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Nombre de Ratings')
axes[0].set_ylabel('Nombre de Films')
axes[0].set_title('Distribution de la Popularité des Films')
axes[0].set_yscale('log')

# Top 20 films
top_movies = ratings.groupby('movieId').size().nlargest(20)
movie_titles = [movies[movies['movieId'] == mid]['title'].values[0] if len(movies[movies['movieId'] == mid]) > 0 else f'Film {mid}' for mid in top_movies.index]
axes[1].barh(range(len(top_movies)), top_movies.values)
axes[1].set_yticks(range(len(top_movies)))
axes[1].set_yticklabels([title[:30] + '...' if len(title) > 30 else title for title in movie_titles], fontsize=8)
axes[1].set_xlabel('Nombre de Ratings')
axes[1].set_title('Top 20 Films')

plt.tight_layout()
plt.show()

print(f"\nPopularité des films:")
print(f"  Min ratings: {movie_popularity.min()}")
print(f"  Max ratings: {movie_popularity.max()}")
print(f"  Moyenne: {movie_popularity.mean():.1f}")
print(f"  Médiane: {movie_popularity.median():.1f}")

## 2. Visualisation du Graphe Biparti

In [ ]:
# Prétraiter les données
interactions = loader.preprocess(min_rating=3.5)

print(f"\nAprès filtrage (rating >= 3.5):")
print(f"  Interactions: {len(interactions)}")
print(f"  Utilisateurs: {interactions['user_id'].nunique()}")
print(f"  Films: {interactions['item_id'].nunique()}")

# Statistiques du graphe
stats = loader.get_stats()
print(f"\nStatistiques du graphe:")
for key, value in stats.items():
    if key != 'density':
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value:.6f}")

In [ ]:
# Analyser la sparsité
fig, ax = plt.subplots(figsize=(10, 6))

sparsity_metric = 1 - stats['density']
labels = ['Interactions', 'Cellules Vides']
sizes = [stats['density'], sparsity_metric]
colors = ['#FF6B6B', '#4ECDC4']

wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.4f%%', colors=colors, startangle=90)
ax.set_title(f'Matrice d\'Interactions Utilisateur-Film\nSparsité: {sparsity_metric:.4f}')

plt.tight_layout()
plt.show()

print(f"\nMatrice d'interactions:")
print(f"  Taille: {stats['n_users']} × {stats['n_items']}")
print(f"  Nombre total de cellules: {stats['n_users'] * stats['n_items']:,}")
print(f"  Interactions observées: {stats['n_interactions']:,}")
print(f"  Densité: {stats['density']:.6f}")
print(f"  Sparsité: {sparsity_metric:.6f}")

## 3. Analyse des Résultats d'Entraînement

In [ ]:
# Charger les logs d'entraînement
import glob

log_files = glob.glob('../results/logs/*.log')
if log_files:
    latest_log = max(log_files, key=os.path.getctime)
    print(f"Fichier de log trouvé: {latest_log}\n")
    
    # Lire les logs
    with open(latest_log, 'r') as f:
        lines = f.readlines()
    
    # Extraire epochs et losses
    epochs = []
    losses = []
    
    for line in lines[3:]:  # Skip header
        if 'Epoch' in line and 'Loss' in line:
            try:
                parts = line.split('Epoch')[-1]
                epoch_num = int(parts.split('/')[0].strip().split('[')[-1])
                loss_val = float(parts.split('Loss:')[-1].strip())
                epochs.append(epoch_num)
                losses.append(loss_val)
            except:
                pass
    
    if epochs:
        print(f"Epochs trouvés: {len(epochs)}")
        print(f"Loss finale: {losses[-1]:.4f}")
        print(f"\nPremiers epochs:")
        for i in range(min(5, len(epochs))):
            print(f"  Epoch {epochs[i]}: Loss = {losses[i]:.4f}")
        
        # Tracer la courbe de loss
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(epochs, losses, marker='o', linestyle='-', linewidth=2, markersize=4)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('BPR Loss')
        ax.set_title('Courbe de Convergence d\'Entraînement')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("Aucun fichier de log trouvé. Exécutez d'abord le modèle avec main.py")

## 4. Résumé et Conclusions

In [ ]:
print("\n" + "="*60)
print("RÉSUMÉ DE L'ANALYSE")
print("="*60)

print(f"\n1. DATASET MOVIESLENS")
print(f"   - Utilisateurs: {ratings['userId'].nunique():,}")
print(f"   - Films: {ratings['movieId'].nunique():,}")
print(f"   - Ratings totaux: {len(ratings):,}")
print(f"   - Rating moyen: {ratings['rating'].mean():.2f}")
print(f"   - Plage: [{ratings['rating'].min()}, {ratings['rating'].max()}]")

print(f"\n2. APRÈS PRÉTRAITEMENT (Rating >= 3.5)")
print(f"   - Interactions: {stats['n_interactions']:,}")
print(f"   - Densité du graphe: {stats['density']:.6f}")
print(f"   - Sparsité: {1-stats['density']:.6f}")

print(f"\n3. CARACTÉRISTIQUES")
print(f"   - Distribution utilisateurs: {user_activity.min()}-{user_activity.max()} ratings")
print(f"   - Distribution films: {movie_popularity.min()}-{movie_popularity.max()} ratings")

print(f"\n4. RECOMMANDATIONS POUR LightGCN")
print(f"   - Embedding dimension: 64")
print(f"   - Nombre de couches: 3")
print(f"   - Batch size: 1024")
print(f"   - Learning rate: 0.001")
print(f"   - Epochs recommandés: 100-200")

print("\n" + "="*60)